In [1]:
import gc
import os
from collections import Counter

import numpy as np
import torch
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

from dataset.pac_matrix import PAC5MatrixHoFDataset
from gcnn.base_model import GroupEquivariantCNN

K_FOLDS = 5
BATCH_SIZE = 128
PRETRAINED_WEIGHTS_PATH = (
    "../models/gcnn_3.7M-pac_grid-[PAC5-1-no_high_fidelity]-fold_1/best.ckpt"
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class GCNNDeltaCorrection(nn.Module):
    def __init__(self, kernel_size=5, num_hidden=5, hidden_channels=48):
        super().__init__()

        self.feature_extractor = GroupEquivariantCNN(
            property_types=["extensive"],
            kernel_size=kernel_size,
            num_hidden=num_hidden,
            hidden_channels=hidden_channels,
        )

        self.feature_extractor.thermodynamic_head = nn.Identity()
        self.feature_extractor.electronic_head = nn.Identity()

        self.correction_head = nn.Sequential(
            nn.BatchNorm1d(hidden_channels + 1),
            nn.Linear(hidden_channels + 1, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 1),
        )

    def forward(self, pac_tensor, low_level_hof):
        if low_level_hof.dim() == 1:
            low_level_hof = low_level_hof.unsqueeze(1)

        _, extensive_features = self.feature_extractor.extract_features(pac_tensor)
        structural_features = extensive_features
        combined = torch.cat([structural_features, low_level_hof], dim=1)

        delta = self.correction_head(combined)
        corrected_hof = low_level_hof + delta

        return corrected_hof, delta


dataset = PAC5MatrixHoFDataset(target_hof_col="nist_hof")

stratify_labels = [f"{item[4]}_{item[5]}" for item in dataset.paired_data]
label_counts = Counter(stratify_labels)

safe_stratify_labels = np.array(
    [
        label if label_counts[label] >= K_FOLDS else "rare_combination"
        for label in stratify_labels
    ]
)
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold_results = []

Using device: cuda
Fetching paired HOF data...
Matched 621 paired samples.


In [2]:
for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), safe_stratify_labels), 1
):
    print(f"\n======[ Fold {fold}/{K_FOLDS} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=safe_stratify_labels[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    log_dir = (
        f"../logs/gcnn_3.7M-pac_grid-[PAC5-hof-to_nist_hof]-finetune_delta/fold_{fold}"
    )
    os.makedirs(log_dir, exist_ok=True)
    writer = SummaryWriter(log_dir=log_dir)

    train_sub = Subset(dataset, train_idx)
    val_sub = Subset(dataset, val_idx)
    test_sub = Subset(dataset, test_idx)

    train_loader = DataLoader(
        train_sub, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True
    )
    val_loader = DataLoader(
        val_sub, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True
    )
    test_loader = DataLoader(
        test_sub, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True
    )

    model = GCNNDeltaCorrection().to(device)

    pretrained_weights = torch.load(PRETRAINED_WEIGHTS_PATH, map_location=device)
    state_dict = pretrained_weights.get("state_dict", pretrained_weights)
    state_dict = {
        k.replace("model.", "").replace("feature_extractor.", ""): v
        for k, v in state_dict.items()
    }

    model.feature_extractor.load_state_dict(state_dict, strict=False)

    criterion = nn.MSELoss()

    optimizer = optim.AdamW(
        [
            {"params": model.feature_extractor.parameters(), "lr": 5e-4},
            {"params": model.correction_head.parameters(), "lr": 5e-3},
        ],
        weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[5e-4, 5e-3], steps_per_epoch=len(train_loader), epochs=100
    )

    save_dir = "../models/gcnn_3.7M-pac_grid-[PAC5-hof-to_nist_hof]-finetune_delta"
    os.makedirs(save_dir, exist_ok=True)
    save_path = f"{save_dir}/fold_{fold}.pth"

    best_val_loss = float("inf")

    for epoch in range(1, 101):
        model.train()
        running_loss = 0.0

        for pac_tensor, low_hof, high_hof, _, _ in train_loader:
            pac_tensor = pac_tensor.to(device)
            low_hof = low_hof.to(device)
            high_hof = high_hof.to(device)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=device.type == "cuda",
            ):
                corrected_hof, _ = model(pac_tensor, low_hof)
                loss = criterion(corrected_hof, high_hof)

            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)

        model.eval()
        avg_val_loss = 0.0

        with torch.no_grad():
            for pac_tensor, low_hof, high_hof, _, _ in val_loader:
                pac_tensor = pac_tensor.to(device)
                low_hof = low_hof.to(device)
                high_hof = high_hof.to(device)

                with torch.autocast(
                    device_type=device.type,
                    dtype=torch.bfloat16,
                    enabled=device.type == "cuda",
                ):
                    corrected_hof, _ = model(pac_tensor, low_hof)
                    avg_val_loss += criterion(corrected_hof, high_hof).item()

        avg_val_loss /= len(val_loader)

        base_lr = optimizer.param_groups[0]["lr"]
        head_lr = optimizer.param_groups[1]["lr"]

        writer.add_scalar("loss/train", avg_train_loss, epoch)
        writer.add_scalar("loss/val", avg_val_loss, epoch)
        writer.add_scalar("hyperparameters/base_lr", base_lr, epoch)
        writer.add_scalar("hyperparameters/head_lr", head_lr, epoch)
        writer.flush()

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            marker = " (best)"
            torch.save(model.state_dict(), save_path)
        else:
            marker = ""

        if epoch % 10 == 0 or epoch == 1 or marker:
            print(
                f"Epoch {epoch:03d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}{marker}"
            )

    print(f"Best Validation Loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)
    writer.close()

    del train_sub, val_sub, test_sub
    del train_loader, val_loader, test_loader
    del model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Loss per fold: {fold_results}")
print(f"Average Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")


======[ Fold 1/5 ]======
Epoch 001 | Train Loss: 1319.1466 | Val Loss: 1103.8319 (best)
Epoch 002 | Train Loss: 1295.2335 | Val Loss: 1091.3219 (best)
Epoch 003 | Train Loss: 1299.3775 | Val Loss: 1083.9020 (best)
Epoch 004 | Train Loss: 1392.3917 | Val Loss: 1078.5175 (best)
Epoch 005 | Train Loss: 1289.5581 | Val Loss: 1072.6392 (best)
Epoch 006 | Train Loss: 1311.6692 | Val Loss: 1062.9110 (best)
Epoch 007 | Train Loss: 1256.8930 | Val Loss: 1048.5339 (best)
Epoch 008 | Train Loss: 1305.9334 | Val Loss: 1018.2534 (best)
Epoch 009 | Train Loss: 1272.4982 | Val Loss: 969.3688 (best)
Epoch 010 | Train Loss: 1133.2870 | Val Loss: 873.9832 (best)
Epoch 011 | Train Loss: 1034.5688 | Val Loss: 709.0477 (best)
Epoch 012 | Train Loss: 780.5900 | Val Loss: 412.0974 (best)
Epoch 013 | Train Loss: 529.2055 | Val Loss: 201.7922 (best)
Epoch 020 | Train Loss: 63.7488 | Val Loss: 193.6990 (best)
Epoch 021 | Train Loss: 59.8501 | Val Loss: 137.3867 (best)
Epoch 022 | Train Loss: 35.3244 | Val Loss

In [3]:
import os
from collections import Counter

import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset


def evaluate_model(model, dataloader, device):
    model.eval()
    all_targets = []
    all_preds = []

    with torch.no_grad():
        for pac_tensor, low_level, high_level, _, _ in dataloader:
            pac_tensor = pac_tensor.to(device)
            low_level = low_level.to(device)

            pred, _ = model(pac_tensor, low_level)

            all_preds.extend(pred.cpu().numpy())
            all_targets.extend(high_level.numpy())

    all_preds = np.array(all_preds).flatten()
    all_targets = np.array(all_targets).flatten()

    errors = all_preds - all_targets
    abs_errors = np.abs(errors)

    mae = np.mean(abs_errors)
    rmsd = np.sqrt(np.mean(errors**2))
    max_err = np.max(abs_errors)
    min_err = np.min(abs_errors)
    r2 = r2_score(all_targets, all_preds)

    return mae, rmsd, max_err, min_err, r2


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluating on device: {device}")

dataset = PAC5MatrixHoFDataset(target_hof_col="nist_hof")

stratify_labels = [f"{item[4]}_{item[5]}" for item in dataset.paired_data]
k_folds = 5
label_counts = Counter(stratify_labels)

safe_stratify_labels = np.array(
    [
        label if label_counts[label] >= k_folds else "rare_combination"
        for label in stratify_labels
    ]
)

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

fold_train_metrics = []
fold_val_metrics = []
fold_test_metrics = []

for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), safe_stratify_labels), 1
):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=safe_stratify_labels[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    train_sub = Subset(dataset, train_idx)
    val_sub = Subset(dataset, val_idx)
    test_sub = Subset(dataset, test_idx)

    train_loader = DataLoader(train_sub, batch_size=256, shuffle=False)
    val_loader = DataLoader(val_sub, batch_size=256, shuffle=False)
    test_loader = DataLoader(test_sub, batch_size=256, shuffle=False)

    model = GCNNDeltaCorrection().to(device)

    load_path = f"../models/gcnn_3.7M-pac_grid-[PAC5-hof-to_nist_hof]-finetune_delta/fold_{fold}.pth"
    try:
        model.load_state_dict(torch.load(load_path, map_location=device))
    except FileNotFoundError:
        print(f"Could not find {load_path}. Skipping this fold.")
        continue

    train_results = evaluate_model(model, train_loader, device)
    val_results = evaluate_model(model, val_loader, device)
    test_results = evaluate_model(model, test_loader, device)

    fold_train_metrics.append(train_results)
    fold_val_metrics.append(val_results)
    fold_test_metrics.append(test_results)

    t_mae, t_rmsd, t_max, t_min, t_r2 = train_results
    print(
        f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
    )

    v_mae, v_rmsd, v_max, v_min, v_r2 = val_results
    print(
        f" + Val -> MAE: {v_mae:.4f}\tRMSD: {v_rmsd:.4f}\tMax: {v_max:.4f}\tMin: {v_min:.4f}\tR^2: {v_r2:.4f}"
    )

    te_mae, te_rmsd, te_max, te_min, te_r2 = test_results
    print(
        f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
    )

if fold_train_metrics and fold_val_metrics and fold_test_metrics:
    print("\n======[ Final 5-fold cross validation results ]======")

    train_metrics_array = np.array(fold_train_metrics)
    val_metrics_array = np.array(fold_val_metrics)
    test_metrics_array = np.array(fold_test_metrics)

    avg_train = np.mean(train_metrics_array, axis=0)
    std_train = np.std(train_metrics_array, axis=0, ddof=1)

    avg_val = np.mean(val_metrics_array, axis=0)
    std_val = np.std(val_metrics_array, axis=0, ddof=1)

    avg_test = np.mean(test_metrics_array, axis=0)
    std_test = np.std(test_metrics_array, axis=0, ddof=1)

    print("Train:")
    print(f" + MAE: {avg_train[0]:.4f} ± {std_train[0]:.4f}")
    print(f" + RMSD: {avg_train[1]:.4f} ± {std_train[1]:.4f}")
    print(f" + Max: {avg_train[2]:.4f} ± {std_train[2]:.4f}")
    print(f" + Min: {avg_train[3]:.4f} ± {std_train[3]:.4f}")
    print(f" + R^2: {avg_train[4]:.4f} ± {std_train[4]:.4f}\n")

    print("Validation:")
    print(f" + MAE: {avg_val[0]:.4f} ± {std_val[0]:.4f}")
    print(f" + RMSD: {avg_val[1]:.4f} ± {std_val[1]:.4f}")
    print(f" + Max: {avg_val[2]:.4f} ± {std_val[2]:.4f}")
    print(f" + Min: {avg_val[3]:.4f} ± {std_val[3]:.4f}")
    print(f" + R^2: {avg_val[4]:.4f} ± {std_val[4]:.4f}\n")

    print("Test:")
    print(f" + MAE: {avg_test[0]:.4f} ± {std_test[0]:.4f}")
    print(f" + RMSD: {avg_test[1]:.4f} ± {std_test[1]:.4f}")
    print(f" + Max: {avg_test[2]:.4f} ± {std_test[2]:.4f}")
    print(f" + Min: {avg_test[3]:.4f} ± {std_test[3]:.4f}")
    print(f" + R^2: {avg_test[4]:.4f} ± {std_test[4]:.4f}")

Evaluating on device: cuda
Fetching paired HOF data...
Matched 621 paired samples.

======[ Evaluating fold 1/5 ]======
 + Train -> MAE: 2.2233	RMSD: 2.8196	Max: 17.5117	Min: 0.0133	R^2: 0.9990
 + Val -> MAE: 4.2105	RMSD: 6.3343	Max: 25.4229	Min: 0.2051	R^2: 0.9949
 + Test -> MAE: 5.2263	RMSD: 8.9565	Max: 61.9253	Min: 0.1875	R^2: 0.9900

======[ Evaluating fold 2/5 ]======
 + Train -> MAE: 1.9915	RMSD: 2.6447	Max: 16.4146	Min: 0.0005	R^2: 0.9991
 + Val -> MAE: 3.4228	RMSD: 4.9734	Max: 16.7758	Min: 0.1266	R^2: 0.9966
 + Test -> MAE: 3.8418	RMSD: 6.8680	Max: 33.2700	Min: 0.0009	R^2: 0.9945

======[ Evaluating fold 3/5 ]======
 + Train -> MAE: 1.6741	RMSD: 2.2680	Max: 14.4180	Min: 0.0010	R^2: 0.9993
 + Val -> MAE: 3.4674	RMSD: 6.8142	Max: 42.8622	Min: 0.0826	R^2: 0.9934
 + Test -> MAE: 2.9873	RMSD: 4.5559	Max: 21.7728	Min: 0.0344	R^2: 0.9973

======[ Evaluating fold 4/5 ]======
 + Train -> MAE: 2.4914	RMSD: 3.7452	Max: 29.1052	Min: 0.0104	R^2: 0.9983
 + Val -> MAE: 3.8615	RMSD: 6.0025	Max